# CodeAlpha Data Analytics Internship
## Task 1: Web Scraping — Book Catalog Extraction

- **Intern**: Om Katkar
- **Target Site**: [Books to Scrape](http://books.toscrape.com/)
- **Libraries**: `requests`, `BeautifulSoup4`, `pandas`

---
### Project Objective
Extract structured e-commerce product listings across multiple paginated web pages, parse HTML elements, convert text metrics into numeric types (currency, ratings), and structure the data into analysis-ready CSV and JSON formats.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin
from IPython.display import display

print("Required libraries loaded successfully!")

### 1. Configuration & Helper Mappings
Define base URL, headers to simulate legitimate user agent requests, and word-to-integer mapping for star ratings.

In [ ]:
BASE_URL = "http://books.toscrape.com/catalogue/page-{}.html"
RATING_MAP = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5}
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

### 2. Multi-Page Scraper Implementation
Iterate through pagination, parse DOM containers (`article.product_pod`), and extract attributes.

In [ ]:
scraped_books = []
MAX_PAGES = 5

for page in range(1, MAX_PAGES + 1):
    page_url = BASE_URL.format(page)
    res = requests.get(page_url, headers=HEADERS, timeout=10)
    if res.status_code != 200:
        print(f"Page {page} returned status {res.status_code}. Stopping.")
        break
        
    soup = BeautifulSoup(res.content, "html.parser")
    articles = soup.find_all("article", class_="product_pod")
    
    for art in articles:
        # Title & URL
        a_tag = art.find("h3").find("a")
        title = a_tag.get("title", a_tag.text.strip())
        prod_link = urljoin(page_url, a_tag.get("href", ""))
        
        # Price (numeric float)
        price_str = art.find("p", class_="price_color").text
        price = float(re.search(r"[\d.]+", price_str).group())
        
        # Star Rating
        rating_tag = art.find("p", class_="star-rating")
        star_classes = [c.lower() for c in rating_tag.get("class", []) if c.lower() in RATING_MAP]
        rating = RATING_MAP[star_classes[0]] if star_classes else 0
        
        # Stock Availability
        availability = art.find("p", class_="instock availability").text.strip()
        
        scraped_books.append({
            "title": title,
            "price_gbp": price,
            "star_rating": rating,
            "availability": availability,
            "product_url": prod_link,
            "page": page
        })
        
    print(f"Extracted {len(articles)} books from Page {page}.")

print(f"\n[+] Total books collected: {len(scraped_books)}")

### 3. DataFrame Conversion & Summary Statistics

In [ ]:
df = pd.DataFrame(scraped_books)
df.head()

In [ ]:
print("Dataset Shape:", df.shape)
print("\nSummary Statistics:")
display(df.describe())

### 4. Ratings & Price Distribution

In [ ]:
print("Rating Counts:")
display(df['star_rating'].value_counts().sort_index())

### 5. Exporting Processed Dataset

In [ ]:
df.to_csv("books_data.csv", index=False)
df.to_json("books_data.json", orient="records", indent=4)
print("[+] Data exported to books_data.csv and books_data.json successfully!")